# ROGII CV vs LB — how well does GroupKFold transfer? (4 real data points)

Everyone says *"trust your CV"*. On this competition I can actually **measure** how much you should trust it: over two days I made 4 leaderboard submissions spanning **LB 14.4 → 9.96**, each backed by an out-of-fold (OOF) score from the *same fixed 5-fold GroupKFold over all 773 train wells*, with zero LB-based tuning in between.

**TL;DR**

1. **Grouped CV transfers.** Public LB ≈ OOF − 0.35 on average; every large OOF improvement (≥ 0.3) showed up on the LB with the right sign and roughly the right size.
2. **But its resolution is finite.** A 0.05 OOF gap between two of my blends *inverted* on the LB. Below ~0.2 RMSE, OOF differences are noise — don't spend submissions resolving them.
3. The public LB is scored on **hidden wells**, not the 3 visible test wells (I verified this the expensive way — see observation 3).

*Companion notebooks:* [what you actually predict (EDA)](https://www.kaggle.com/code/yujiyyy/rogii-eda-what-you-predict) · [the train copies are NOT the answers](https://www.kaggle.com/code/yujiyyy/rogii-train-copies-are-not-answers) · [honest GroupKFold baseline](https://www.kaggle.com/code/yujiyyy/rogii-honest-groupkfold-baseline)

## 1. Why CV is the only compass in this competition

- **The 3 visible test wells are example data.** At scoring time your notebook is re-run against hidden wells that are not in `train/` ([staff confirmation](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/697507)).
- **Rows within a well are massively autocorrelated.** A random row split leaks and reports fantasy scores; the unit of generalization is the *well*, so folds must hold out entire wells (GroupKFold by well id).
- **Submissions are expensive** (code competition, manual attach). You want to know *in advance* whether a CV gain is worth one.

So the practical question is: *if my grouped OOF improves by X, what happens on the LB?* Below is what actually happened.

In [1]:
import glob, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# context: what the metric is computed on (works on Kaggle; falls back to a local copy)
cands = (glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
         or glob.glob('../../data/sample_submission.csv'))
sample = pd.read_csv(cands[0])
wells = sample['id'].str.rsplit('_', n=1).str[0]
print(f'scored rows: {len(sample):,}  |  example test wells: {wells.nunique()}')
print(wells.value_counts().to_string())

## 2. The experiment: 4 submissions, one fixed CV, no LB tuning

Protocol (kept boring on purpose):

- **Folds fixed once**: 5-fold GroupKFold over all 773 train wells, never changed between experiments.
- **OOF RMSE** computed on exactly the scored region (the rows where `TVT_input` is NaN), same as the LB metric.
- Each submission was selected **by OOF only**, submitted once, and the LB score recorded. No submission was ever chosen or tuned by looking at the LB.

The four submissions are successive GBDT blends with progressively richer feature sets (the modelling details don't matter for this notebook — the point is the *pairs*):

In [2]:
pairs = pd.DataFrame([
    # label,                          OOF (GroupKFold, 773 wells), public LB
    ('sub 1: GBDT blend',                          14.443, 14.353),
    ('sub 2: + per-well signal features',          12.369, 11.644),
    ('sub 3: + richer feature set',                10.343,  9.961),
    ('sub 4: variant of sub 3 (0.05 better OOF)',  10.290, 10.068),
], columns=['submission', 'oof', 'lb'])
pairs['lb_minus_oof'] = (pairs['lb'] - pairs['oof']).round(3)
print(pairs.to_string(index=False))
print(f"\nmean offset (LB - OOF): {pairs['lb_minus_oof'].mean():+.3f}")
print(f"offset range:            {pairs['lb_minus_oof'].min():+.3f} .. {pairs['lb_minus_oof'].max():+.3f}")

In [3]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
lo, hi = 9.0, 15.5
ax.plot([lo, hi], [lo, hi], '--', color='gray', lw=1, label='LB = OOF')
off = pairs['lb_minus_oof'].mean()
ax.plot([lo, hi], [lo + off, hi + off], ':', color='tab:orange', lw=1.5,
        label=f'LB = OOF {off:+.2f} (mean offset)')
ax.scatter(pairs['oof'], pairs['lb'], s=90, color='tab:blue', zorder=3)
for _, r in pairs.iterrows():
    ax.annotate(r['submission'].split(':')[0], (r['oof'], r['lb']),
                textcoords='offset points', xytext=(10, -4), fontsize=10)
ax.set_xlabel('OOF RMSE (5-fold GroupKFold by well, 773 wells)')
ax.set_ylabel('Public LB RMSE (hidden wells)')
ax.set_title('ROGII: grouped CV vs public LB — 4 untuned submissions')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
# inset: zoom on the rank inversion between sub 3 and sub 4
axi = fig.add_axes([0.58, 0.18, 0.3, 0.28])
z = pairs.iloc[2:4]
axi.plot([10.2, 10.4], [10.2, 10.4], '--', color='gray', lw=1)
axi.scatter(z['oof'], z['lb'], s=70, color='tab:red', zorder=3)
for _, r in z.iterrows():
    axi.annotate(r['submission'].split(':')[0], (r['oof'], r['lb']),
                 textcoords='offset points', xytext=(6, -3), fontsize=9)
axi.set_title('rank inversion: 0.05 OOF gap', fontsize=9)
axi.tick_params(labelsize=8)
axi.grid(alpha=0.3)
plt.show()

## 3. Three observations

**① Calibration is tight for real improvements.** Every OOF gain ≥ 0.3 transferred to the LB with the right sign and similar magnitude. The LB sits slightly *below* OOF (mean −0.35), i.e. the hidden public wells score a touch easier than the average of the 773 train wells — a constant-ish offset, not a re-ranking. If your grouped OOF drops by 1.0, expect the LB to drop by roughly 1.0.

**② The resolution limit is ~0.2 RMSE.** Sub 4 had a 0.05 *better* OOF than sub 3, but scored 0.11 *worse* on the LB. The public LB is computed on a finite set of hidden wells, and per-well errors here are heavy-tailed (a few wells with long dip excursions dominate the squared error), so both OOF and LB carry sampling noise of this order. Practical rule: treat OOF differences below ~0.2 as ties, and don't burn a submission to resolve them.

**③ The off-scale control: copying train values scores 15.883.** The 3 visible test wells exist verbatim in `train/` with a full `TVT` column. Submitting those values — what would be a perfect 0.0 if the LB used the visible wells — scores *worse than a constant baseline*. That's the cleanest possible proof that the LB lives on hidden wells, which is exactly *why* grouped CV calibrates so well. Details in [the companion notebook](https://www.kaggle.com/code/yujiyyy/rogii-train-copies-are-not-answers).

One honest caveat: 4 points is a small sample, and the final standings are computed on a *different* (larger?) hidden set than the public LB, so the −0.35 offset itself may shift at rescore time. The structural conclusions (sign transfers, ~0.2 noise floor) are the robust part.

## 4. The playbook this implies

1. **Fix your folds once** (GroupKFold by well), compute OOF on the scored region only, and never change the folds mid-stream — otherwise your own history stops being comparable.
2. **Submit only when OOF improves by ≥ 0.2–0.3.** Smaller gains are indistinguishable from noise on the LB anyway.
3. **Select your final submissions by OOF, not by public LB.** The final rescore runs on yet another hidden well set; public-LB-tuned pipelines are the ones most exposed to a shake-up.
4. A small *constant* CV-vs-LB offset is normal and harmless — watch the *deltas*, not the absolute gap.

If this calibrates your expectations before you spend a submission, an upvote helps others find it. Good luck out there! 🛢️